In [25]:
import os
from dotenv import load_dotenv 
load_dotenv()

os.environ["OPENAI_API _KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [26]:
from langchain.chat_models import init_chat_model

model=init_chat_model("groq:qwen/qwen3-32b")

In [30]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movies was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="the movies rating out of 10")

In [31]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000201CD318410>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000201CD318E10>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movies was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'the movies rating ou

In [32]:
response=model_with_structure.invoke("Provide details about the movies Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [33]:
class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There\'s a Movie function that requires title, year, director, and rating. I need to make sure I have all that information for Inception.\n\nFirst, the title is obviously "Inception". The year it was released is 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yep, 8.8/10. So all the required parameters are there. I\'ll structure the tool call with these details. Make sure the JSON is correctly formatted with the right data types: title and director as strings, year as an integer, and rating as a number. No need for any other functions here since the user just wants the movie details. Alright, that should cover it.\n', 'tool_calls': [{'id': 'qw7q26we2', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","ye

In [34]:
from pydantic import BaseModel

class Actor(BaseModel):
    name:str
    role:str

class Movie(BaseModel):
    title:str
    year:str
    cast:list[Actor]
    genres:list[str]
    budget:float| None=Field(None, description="Budget in millions USD")

model_with_structure=model.with_structured_output(Movie)

response=model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year='2010', cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Mal Cobb'), Actor(name='Tom Hardy', role="Cobb's Forger")], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

In [36]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent

class contactinfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent=create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=contactinfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='83d3f35c-24d2-4054-ba7c-6e0f69f753af'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants me to extract contact information from the given text: "John Doe, john@example.com, (555) 123-4567". Let me look at the tools provided. There\'s a function called contactinfo with parameters name, email, and phone. All three are required.\n\nFirst, I need to parse the input. The name is John Doe. The email is clearly john@example.com. The phone number is (555) 123-4567. I should check if the phone number is in the correct format. The function expects a string for each parameter. So I\'ll map each part to the respective parameter. Name: John Doe, Email: john@example.com, Phone: (555) 123-4567. No need to reformat the phone number unless specified, and the user didn\'t mention any formatting changes. So I\'ll j

In [37]:
result["structured_response"]

contactinfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')